# 01 · Phát hiện chép câu trả lời LLM bằng nhịp gõ phím (song ngữ Việt – Anh)

Notebook này phục vụ **C1** (phương thức keystroke trong mô hình hợp nhất) và chuẩn bị dữ liệu cho notebook 03, 04.

**Dữ liệu (tự tải):**
| Bộ | Ngôn ngữ | Nhãn | Nguồn |
|---|---|---|---|
| Dong et al., ICTAI 2026 (arXiv 2610.07700) | Tiếng Việt, 45 người | B (tự viết), P (diễn đạt lại câu trả lời LLM), T (chép LLM), FP/FT (P/T có thao túng nhịp gõ) | Google Drive của tác giả, `github.com/dongphuthanh/Vietnamese_keystrokes` |
| Kundu et al., IJCB 2024 (arXiv 2406.15335) | Tiếng Anh, 40 người + SBU + Buffalo | free (tự viết) / fixed (chép) | `github.com/ijcb-2024/keystroke-llm-plagiarism` (Git LFS) |

Hãy kiểm tra giấy phép của từng bộ dữ liệu trước khi dùng cho bài báo (README của hai repo không ghi license).

**Đặc trưng:** dùng ĐÚNG công thức mà trình duyệt thí sinh tính (cửa sổ 2 giây, chuẩn hóa theo đoạn khởi động 60 giây,
lượng tử hóa + nhiễu Laplace) — xem `eg_common.py`. Nhờ vậy mô hình huấn luyện ở đây chạy được trên dữ liệu thật của nền tảng.

**Thí nghiệm:**
1. Baseline XGBoost trên đặc trưng tổng hợp mỗi câu trả lời.
2. Mô hình chuỗi (TCN nhỏ) trên chuỗi cửa sổ 2 giây.
3. Huấn luyện riêng từng ngôn ngữ / chung hai ngôn ngữ / chéo ngôn ngữ.
4. Chống né tránh: huấn luyện trên B, P, T rồi kiểm tra trên FP, FT (người gian lận cố gõ giống tự nhiên).

Quy tắc: **chia theo người** (không người nào vừa ở tập huấn luyện vừa ở tập kiểm tra); báo cáo AUC, recall @ FPR 1% / 5%.

Thời gian chạy: ~10–20 phút trên Colab (CPU đủ; GPU giúp phần TCN nhanh hơn).

In [ ]:
!pip -q install gdown xgboost
!apt-get -qq install -y git-lfs > /dev/null && git lfs install --skip-repo > /dev/null

### Mã dùng chung
Hai ô dưới ghi `feature_schema.json` (hợp đồng đặc trưng với trang web) và `eg_common.py` (công thức đặc trưng, baseline luật, mô hình hợp nhất). Không cần sửa.

In [ ]:
%%writefile feature_schema.json
{
 "version": 1,
 "window_seconds": 2,
 "modalities": [
  "face",
  "identity",
  "object",
  "audio",
  "browser",
  "keystroke",
  "answer",
  "system"
 ],
 "features": [
  {
   "name": "face_count_mean",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Số khuôn mặt trung bình"
  },
  {
   "name": "face_count_max",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Số khuôn mặt lớn nhất"
  },
  {
   "name": "face_missing_ratio",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ mẫu không thấy mặt"
  },
  {
   "name": "yaw_mean",
   "modality": "face",
   "scale": 45,
   "source": "client",
   "desc": "Góc quay ngang trung bình (độ, ước lượng từ 6 điểm)"
  },
  {
   "name": "yaw_std",
   "modality": "face",
   "scale": 20,
   "source": "client",
   "desc": "Độ lệch chuẩn góc quay ngang"
  },
  {
   "name": "pitch_mean",
   "modality": "face",
   "scale": 30,
   "source": "client",
   "desc": "Góc ngẩng/cúi trung bình (độ)"
  },
  {
   "name": "pitch_std",
   "modality": "face",
   "scale": 15,
   "source": "client",
   "desc": "Độ lệch chuẩn góc ngẩng/cúi"
  },
  {
   "name": "head_away_ratio",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ mẫu quay đầu quá ngưỡng (|yaw|>30° hoặc |pitch|>25°)"
  },
  {
   "name": "box_shift",
   "modality": "face",
   "scale": 0.2,
   "source": "client",
   "desc": "Dịch chuyển tâm hộp mặt (theo chiều rộng khung)"
  },
  {
   "name": "face_size",
   "modality": "face",
   "scale": 0.3,
   "source": "client",
   "desc": "Diện tích hộp mặt / diện tích khung"
  },
  {
   "name": "face_conf",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Độ tin cậy dò mặt trung bình"
  },
  {
   "name": "id_score",
   "modality": "identity",
   "scale": 1,
   "source": "server",
   "desc": "Điểm khớp gần nhất với ảnh đăng ký"
  },
  {
   "name": "id_checked",
   "modality": "identity",
   "scale": 1,
   "source": "server",
   "desc": "Có lần xác thực lại trong cửa sổ"
  },
  {
   "name": "id_mismatch",
   "modality": "identity",
   "scale": 1,
   "source": "server",
   "desc": "Lần xác thực gần nhất không khớp"
  },
  {
   "name": "obj_phone",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất điện thoại"
  },
  {
   "name": "obj_book",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất sách/giấy"
  },
  {
   "name": "obj_headphone",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất tai nghe"
  },
  {
   "name": "obj_person_extra",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất có người thứ hai (theo thân người)"
  },
  {
   "name": "obj_screen",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất màn hình/laptop phụ"
  },
  {
   "name": "obj_age",
   "modality": "object",
   "scale": 30,
   "source": "server",
   "desc": "Số giây kể từ khung gần nhất (độ tươi)"
  },
  {
   "name": "speech_ratio",
   "modality": "audio",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ thời gian có tiếng nói"
  },
  {
   "name": "speech_segments",
   "modality": "audio",
   "scale": 3,
   "source": "client",
   "desc": "Số đoạn nói bắt đầu trong cửa sổ"
  },
  {
   "name": "audio_energy",
   "modality": "audio",
   "scale": 1,
   "source": "client",
   "desc": "Năng lượng âm thanh (0..1, thang dB chuẩn hóa)"
  },
  {
   "name": "hidden_sec",
   "modality": "browser",
   "scale": 2,
   "source": "client",
   "desc": "Số giây trang bị ẩn (rời tab)"
  },
  {
   "name": "blur_sec",
   "modality": "browser",
   "scale": 2,
   "source": "client",
   "desc": "Số giây cửa sổ mất focus"
  },
  {
   "name": "tab_switches",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần rời tab/cửa sổ"
  },
  {
   "name": "fullscreen_exits",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần thoát toàn màn hình"
  },
  {
   "name": "not_fullscreen_sec",
   "modality": "browser",
   "scale": 2,
   "source": "client",
   "desc": "Số giây không ở chế độ toàn màn hình"
  },
  {
   "name": "window_small",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Cửa sổ bị thu nhỏ/chia đôi (0/1)"
  },
  {
   "name": "screen_extended",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Đang có màn hình mở rộng (0/1, chỉ Chromium)"
  },
  {
   "name": "screen_changes",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần thay đổi cấu hình màn hình"
  },
  {
   "name": "paste_attempts",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần cố dán"
  },
  {
   "name": "copy_attempts",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần cố sao chép/cắt đề"
  },
  {
   "name": "drop_attempts",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần cố kéo thả"
  },
  {
   "name": "contextmenu",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần mở menu chuột phải"
  },
  {
   "name": "chars_after_return",
   "modality": "browser",
   "scale": 20,
   "source": "client",
   "desc": "Số ký tự gõ trong 10 giây ngay sau khi quay lại tab"
  },
  {
   "name": "keys",
   "modality": "keystroke",
   "scale": 10,
   "source": "client",
   "desc": "Số phím (đã trừ phím do bộ gõ sinh)"
  },
  {
   "name": "dwell_rel",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Thời gian giữ phím TB / mức khởi động"
  },
  {
   "name": "dwell_cv",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Hệ số biến thiên thời gian giữ phím"
  },
  {
   "name": "flight_rel",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Khoảng cách giữa hai phím TB / mức khởi động"
  },
  {
   "name": "flight_cv",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Hệ số biến thiên khoảng cách giữa hai phím"
  },
  {
   "name": "burst_ratio",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ khoảng cách < 150 ms"
  },
  {
   "name": "pause_ratio",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ khoảng cách > 2 giây"
  },
  {
   "name": "backspace_ratio",
   "modality": "keystroke",
   "scale": 0.3,
   "source": "client",
   "desc": "Tỉ lệ phím xóa/sửa"
  },
  {
   "name": "compositions",
   "modality": "keystroke",
   "scale": 3,
   "source": "client",
   "desc": "Số lần bộ gõ IME ghép chữ"
  },
  {
   "name": "chars_per_sec",
   "modality": "keystroke",
   "scale": 5,
   "source": "client",
   "desc": "Ký tự/giây"
  },
  {
   "name": "question_switches",
   "modality": "answer",
   "scale": 1,
   "source": "client",
   "desc": "Số lần chuyển câu"
  },
  {
   "name": "answer_changes",
   "modality": "answer",
   "scale": 1,
   "source": "client",
   "desc": "Số lần đổi đáp án"
  },
  {
   "name": "time_on_question",
   "modality": "answer",
   "scale": 60,
   "source": "client",
   "desc": "Số giây đã ở câu hiện tại"
  },
  {
   "name": "progress",
   "modality": "answer",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ câu đã trả lời"
  },
  {
   "name": "time_vs_expected",
   "modality": "answer",
   "scale": 2,
   "source": "client",
   "desc": "(thời gian trả lời - kỳ vọng theo độ khó) / độ lệch, âm = nhanh bất thường"
  },
  {
   "name": "lag_ms",
   "modality": "system",
   "scale": 200,
   "source": "client",
   "desc": "Độ trễ khung hình giao diện làm bài (watchdog)"
  },
  {
   "name": "offline",
   "modality": "system",
   "scale": 1,
   "source": "client",
   "desc": "Mất kết nối trong cửa sổ (0/1, không coi là gian lận)"
  },
  {
   "name": "monitor_load",
   "modality": "system",
   "scale": 0.2,
   "source": "client",
   "desc": "Tỉ lệ thời gian CPU dành cho giám sát"
  }
 ],
 "model_input": "concat(clip(x/scale,-5,5) cho từng đặc trưng, mask theo phương thức 0/1) -> 54 + 8 = 62 chiều",
 "privacy": {
  "keystroke_laplace_epsilon": 2.0,
  "keystroke_quantize": 0.05,
  "warmup_seconds": 60
 }
}

In [ ]:
%%writefile eg_common.py
"""Mã dùng chung cho các notebook ExamGuard (được nhúng vào notebook bằng %%writefile để chạy độc lập trên Colab).

Các hàm đặc trưng ở đây là bản Python của frontend/src/monitor/features.ts — PHẢI giữ khớp công thức,
vì mô hình huấn luyện ở notebook sẽ chạy trên đặc trưng do trình duyệt thí sinh tính.
"""
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np

# ----------------------------------------------------------------------------- schema
_HERE = Path(__file__).resolve().parent
SCHEMA_PATH = next(p for p in (_HERE / "feature_schema.json", _HERE.parent / "shared" / "feature_schema.json") if p.exists())
SCHEMA = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
FEATURES = [f["name"] for f in SCHEMA["features"]]
MODALITIES = SCHEMA["modalities"]
SCALES = np.array([float(f["scale"]) for f in SCHEMA["features"]], np.float32)
MOD_OF = {f["name"]: f["modality"] for f in SCHEMA["features"]}
MOD_SLICES = {}
for _m in MODALITIES:
    _idx = [i for i, f in enumerate(SCHEMA["features"]) if f["modality"] == _m]
    MOD_SLICES[_m] = slice(_idx[0], _idx[-1] + 1)
DIM = len(FEATURES) + len(MODALITIES)
WINDOW_S = SCHEMA["window_seconds"]


def model_input(values: dict, masks: dict) -> np.ndarray:
    """Giống backend FeatureSpec.model_input: clip(x/scale) * mask + mask."""
    x = np.array([float(values.get(n, 0.0) or 0.0) for n in FEATURES], np.float32)
    x = np.clip(x / SCALES, -5, 5)
    m = np.array([1.0 if masks.get(mod) else 0.0 for mod in MODALITIES], np.float32)
    for k, mod in enumerate(MODALITIES):
        x[MOD_SLICES[mod]] *= m[k]
    return np.concatenate([x, m]).astype(np.float32)


# ----------------------------------------------------------------------------- khuôn mặt
def head_pose(kps) -> tuple[float, float]:
    """kps: 6 điểm BlazeFace chuẩn hóa [(x,y)...]: mắt phải, mắt trái, mũi, miệng, tai phải, tai trái."""
    if kps is None or len(kps) < 4:
        return 0.0, 0.0
    re, le, nose, mouth = [np.asarray(k[:2], float) for k in kps[:4]]
    mid = (re + le) / 2
    eye = max(float(np.hypot(*(le - re))), 1e-6)
    roll = math.atan2(le[1] - re[1], le[0] - re[0])
    c, s = math.cos(-roll), math.sin(-roll)

    def rot(p):
        d = p - mid
        return np.array([d[0] * c - d[1] * s, d[0] * s + d[1] * c])

    n, m = rot(nose), rot(mouth)
    yaw = math.degrees(math.atan(n[0] / eye / 0.6))
    pitch = (n[1] / max(m[1], 1e-6) - 0.55) * 120
    return float(np.clip(yaw, -90, 90)), float(np.clip(pitch, -60, 60))


def face_window(samples: list[dict]) -> dict:
    """samples: [{count, yaw, pitch, cx, cy, size, conf}] trong một cửa sổ 2 giây."""
    n = len(samples) or 1
    pres = [s for s in samples if s["count"] > 0]
    yaws = [s["yaw"] for s in pres]
    pit = [s["pitch"] for s in pres]
    shift = sum(math.hypot(pres[i]["cx"] - pres[i - 1]["cx"], pres[i]["cy"] - pres[i - 1]["cy"]) for i in range(1, len(pres)))
    sd = lambda a: float(np.std(a, ddof=1)) if len(a) > 1 else 0.0  # noqa: E731
    return {
        "face_count_mean": float(np.mean([s["count"] for s in samples])) if samples else 0.0,
        "face_count_max": max([s["count"] for s in samples], default=0),
        "face_missing_ratio": (len(samples) - len(pres)) / n if samples else 0.0,
        "yaw_mean": float(np.mean(yaws)) if yaws else 0.0, "yaw_std": sd(yaws),
        "pitch_mean": float(np.mean(pit)) if pit else 0.0, "pitch_std": sd(pit),
        "head_away_ratio": (sum(1 for s in pres if abs(s["yaw"]) > 30 or abs(s["pitch"]) > 25) / len(pres)) if pres else 0.0,
        "box_shift": shift,
        "face_size": float(np.mean([s["size"] for s in pres])) if pres else 0.0,
        "face_conf": float(np.mean([s["conf"] for s in pres])) if pres else 0.0,
    }


# ----------------------------------------------------------------------------- nhịp gõ
def key_timings(events: list[tuple]) -> tuple[list, list, int, int]:
    """events: [(t_ms, 'down'|'up', code, ime_bool)] -> dwell, flight, số phím, số phím sửa."""
    down, dwell, flight = {}, [], []
    last, keys, edits = None, 0, 0
    for t, typ, code, ime in events:
        if ime:
            continue
        if typ == "down":
            if code in down:
                continue
            down[code] = t
            keys += 1
            if code in ("Backspace", "Delete"):
                edits += 1
            if last is not None:
                flight.append(t - last)
            last = t
        else:
            t0 = down.pop(code, None)
            if t0 is not None:
                dwell.append(t - t0)
    return [d for d in dwell if 0 < d < 2000], [f for f in flight if 0 < f < 30000], keys, edits


def keystroke_window(events, compositions: int, inserted_chars: int, seconds: float, base: dict | None) -> dict:
    dwell, flight, keys, edits = key_timings(events)
    fl = [f for f in flight if f < 5000]
    dm = float(np.mean(dwell)) if dwell else 0.0
    fm = float(np.mean(fl)) if fl else 0.0
    sd = lambda a: float(np.std(a, ddof=1)) if len(a) > 1 else 0.0  # noqa: E731
    return {
        "keys": keys,
        "dwell_rel": (dm / base["dwell"] if base and dm else (1.0 if dm else 0.0)),
        "dwell_cv": sd(dwell) / dm if dm else 0.0,
        "flight_rel": (fm / base["flight"] if base and fm else (1.0 if fm else 0.0)),
        "flight_cv": sd(fl) / fm if fm else 0.0,
        "burst_ratio": sum(f < 150 for f in flight) / len(flight) if flight else 0.0,
        "pause_ratio": sum(f > 2000 for f in flight) / len(flight) if flight else 0.0,
        "backspace_ratio": edits / keys if keys else 0.0,
        "compositions": compositions,
        "chars_per_sec": inserted_chars / max(seconds, 1e-3),
    }


PRIVATE_KEYSTROKE = ["dwell_rel", "dwell_cv", "flight_rel", "flight_cv", "burst_ratio", "pause_ratio"]


def privatize(v: dict, epsilon: float, step: float = 0.05, rng: np.random.Generator | None = None) -> dict:
    """Lượng tử hóa + nhiễu Laplace(b = step/ε) như trên máy thí sinh. ε = inf -> chỉ lượng tử hóa."""
    rng = rng or np.random.default_rng()
    out = dict(v)
    for k in PRIVATE_KEYSTROKE:
        if k in out:
            noise = 0.0 if not np.isfinite(epsilon) else rng.laplace(0, step / max(epsilon, 1e-6))
            out[k] = max(0.0, round((out[k] + noise) / step) * step)
    return out


_NON_CHAR = {"backspace", "delete", "shift", "shiftleft", "shiftright", "control", "controlleft", "controlright", "ctrl",
             "alt", "altleft", "altright", "meta", "metaleft", "metaright", "tab", "enter", "capslock", "escape", "esc",
             "arrowup", "arrowdown", "arrowleft", "arrowright", "home", "end", "pageup", "pagedown", "insert"}


def is_char_key(code) -> bool:
    c = str(code).lower()
    return c not in _NON_CHAR and not c.startswith(("f1", "f2", "f3", "f4", "f5", "f6", "f7", "f8", "f9"))


def session_keystroke_windows(events: list[tuple], warmup_s: float = 60, epsilon: float = float("inf"),
                              window_s: float = WINDOW_S, rng=None) -> list[dict]:
    """Cắt chuỗi phím (t_ms tuyệt đối) thành cửa sổ 2 giây với chuẩn hóa theo đoạn khởi động, như trình duyệt."""
    if not events:
        return []
    events = sorted(events, key=lambda e: e[0])
    t0 = events[0][0]
    base, warm_d, warm_f = None, [], []
    out, cur, k = [], [], 0
    end = events[-1][0]
    w_start = t0
    i = 0
    while w_start <= end:
        w_end = w_start + window_s * 1000
        cur = []
        while i < len(events) and events[i][0] < w_end:
            cur.append(events[i])
            i += 1
        if base is None:
            d, f, _, _ = key_timings(cur)
            warm_d += d
            warm_f += [x for x in f if x < 5000]
            if (w_end - t0) / 1000 >= warmup_s and len(warm_d) >= 30 and len(warm_f) >= 30:
                base = {"dwell": float(np.median(warm_d)), "flight": float(np.median(warm_f))}
        chars = sum(1 for e in cur if e[1] == "down" and not e[3] and is_char_key(e[2]))
        w = keystroke_window(cur, 0, chars, window_s, base)
        w = privatize(w, epsilon, rng=rng) if epsilon is not None else w
        d, f, _, _ = key_timings(cur)
        # thời gian tuyệt đối (ms): chỉ dùng để đánh giá rủi ro tái nhận dạng ở notebook 04 — trình duyệt KHÔNG gửi
        w["_dwell_ms"] = float(np.mean(d)) if d else 0.0
        w["_flight_ms"] = float(np.mean([x for x in f if x < 5000])) if f else 0.0
        w["_dwell_ms_std"] = float(np.std(d)) if len(d) > 1 else 0.0
        out.append(w)
        w_start = w_end
        k += 1
    return out


# ----------------------------------------------------------------------------- âm thanh
def audio_window(speech_flags: list[bool], rms: list[float]) -> dict:
    onsets = sum(1 for i, s in enumerate(speech_flags) if s and (i == 0 or not speech_flags[i - 1]))
    r = float(np.mean(rms)) if rms else 0.0
    return {"speech_ratio": float(np.mean(speech_flags)) if speech_flags else 0.0, "speech_segments": onsets,
            "audio_energy": float(np.clip((20 * math.log10(r) + 60) / 60, 0, 1)) if r > 0 else 0.0}


def time_vs_expected(seconds: float, expected: float) -> float:
    return float(np.clip((seconds - expected) / (0.5 * expected), -5, 5)) if expected else 0.0


# ----------------------------------------------------------------------------- baseline luật ngưỡng (khớp backend)
RULES = [
    ("MULTI_FACE", "face", 0.60, lambda v: v["face_count_max"] >= 2),
    ("NO_FACE", "face", 0.25, lambda v: v["face_missing_ratio"] >= 0.99),
    ("HEAD_AWAY", "face", 0.15, lambda v: v["head_away_ratio"] >= 0.5),
    ("ID_MISMATCH", "identity", 0.70, lambda v: v["id_mismatch"] >= 1),
    ("PHONE", "object", 0.60, lambda v: v["obj_phone"] >= 0.5),
    ("PERSON_EXTRA", "object", 0.50, lambda v: v["obj_person_extra"] >= 0.5),
    ("HEADPHONE", "object", 0.35, lambda v: v["obj_headphone"] >= 0.5),
    ("BOOK", "object", 0.30, lambda v: v["obj_book"] >= 0.5),
    ("SCREEN", "object", 0.30, lambda v: v["obj_screen"] >= 0.5),
    ("SPEECH", "audio", 0.20, lambda v: v["speech_ratio"] >= 0.3),
    ("TAB_AWAY", "browser", 0.40, lambda v: v["hidden_sec"] > 0 or v["tab_switches"] > 0),
    ("BLUR", "browser", 0.20, lambda v: v["blur_sec"] >= 1),
    ("FULLSCREEN_EXIT", "browser", 0.20, lambda v: v["fullscreen_exits"] > 0 or v["not_fullscreen_sec"] >= 2),
    ("WINDOW_SMALL", "browser", 0.15, lambda v: v["window_small"] >= 1),
    ("EXTENDED_SCREEN", "browser", 0.20, lambda v: v["screen_extended"] >= 1),
    ("PASTE", "browser", 0.40, lambda v: v["paste_attempts"] > 0 or v["drop_attempts"] > 0),
    ("COPY", "browser", 0.25, lambda v: v["copy_attempts"] > 0),
    ("TYPE_AFTER_RETURN", "browser", 0.40, lambda v: v["chars_after_return"] >= 20),
    ("TRANSCRIBE_RHYTHM", "keystroke", 0.20, lambda v: v["keys"] >= 10 and v["pause_ratio"] < 0.02
     and v["backspace_ratio"] < 0.02 and v["burst_ratio"] > 0.6 and v["flight_cv"] < 0.5),
    ("FAST_ANSWER", "answer", 0.15, lambda v: v["time_vs_expected"] <= -2.5),
]


def rule_score(values: dict, masks: dict) -> float:
    v = {n: float(values.get(n, 0.0)) for n in FEATURES}
    p = 1.0
    for _, mod, w, test in RULES:
        if masks.get(mod) and test(v):
            p *= 1 - w
    return 1 - p


def rule_score_matrix(X: np.ndarray) -> np.ndarray:
    """X: [N, DIM] (đầu vào mô hình) -> điểm luật. Giải chuẩn hóa bằng scale."""
    out = np.zeros(len(X), np.float32)
    for i, x in enumerate(X):
        vals = dict(zip(FEATURES, (x[:len(FEATURES)] * SCALES).tolist()))
        masks = {m: x[len(FEATURES) + k] > 0.5 for k, m in enumerate(MODALITIES)}
        out[i] = rule_score(vals, masks)
    return out


# ----------------------------------------------------------------------------- đánh giá
def recall_at_fpr(y: np.ndarray, s: np.ndarray, fpr: float) -> float:
    y, s = np.asarray(y).astype(bool), np.asarray(s, float)
    neg = np.sort(s[~y])[::-1]
    if not len(neg) or not y.any():
        return float("nan")
    k = int(np.floor(fpr * len(neg)))
    thr = neg[k] if k < len(neg) else -np.inf
    return float((s[y] > thr).mean())


def summarize(y, s) -> dict:
    from sklearn.metrics import average_precision_score, roc_auc_score

    y = np.asarray(y).astype(int)
    if y.min() == y.max():
        return {"n": len(y), "pos": int(y.sum())}
    return {"n": len(y), "pos": int(y.sum()), "auc": round(roc_auc_score(y, s), 4), "ap": round(average_precision_score(y, s), 4),
            "recall@fpr1%": round(recall_at_fpr(y, s, 0.01), 4), "recall@fpr5%": round(recall_at_fpr(y, s, 0.05), 4)}


# ----------------------------------------------------------------------------- mô hình hợp nhất (PyTorch)
def build_fusion_model(hidden: int = 64, d: int = 32, dropout: float = 0.1):
    import torch
    from torch import nn

    class FusionGRU(nn.Module):
        """Encoder riêng từng phương thức -> cổng (gated fusion, che phương thức thiếu) -> GRU nhân quả -> xác suất.

        Một bước = một cửa sổ 2 giây; trạng thái ẩn h mang thông tin quá khứ (streaming)."""

        def __init__(self):
            super().__init__()
            self.slices = [MOD_SLICES[m] for m in MODALITIES]
            self.enc = nn.ModuleList([nn.Sequential(nn.Linear(s.stop - s.start, d), nn.GELU(), nn.Linear(d, d))
                                      for s in self.slices])
            self.gate = nn.Linear(len(MODALITIES) * d, len(MODALITIES))
            self.drop = nn.Dropout(dropout)
            self.gru = nn.GRUCell(d, hidden)
            self.head = nn.Sequential(nn.Linear(hidden + d, hidden), nn.GELU(), nn.Linear(hidden, 1))
            self.hidden = hidden
            self.recon = nn.Linear(hidden + d, len(FEATURES))  # cho tiền huấn luyện tự giám sát

        def fuse(self, x):
            feats = x[..., :len(FEATURES)]
            mask = x[..., len(FEATURES):]
            z = torch.stack([enc(feats[..., s]) for enc, s in zip(self.enc, self.slices)], dim=-2)  # [..., M, d]
            logits = self.gate(z.flatten(-2)) + (mask - 1) * 1e4  # phương thức thiếu -> cổng = 0
            g = torch.softmax(logits, dim=-1)
            g = g * (mask.sum(-1, keepdim=True) > 0)
            fused = (g.unsqueeze(-1) * z).sum(-2)
            return self.drop(fused), g

        def step(self, x, h):
            fused, g = self.fuse(x)
            h2 = self.gru(fused, h)
            out = torch.cat([h2, fused], -1)
            return torch.sigmoid(self.head(out)).squeeze(-1), h2, g

        def forward(self, xs):  # xs: [B, T, D] -> logits [B, T], gates [B, T, M], recon [B, T, F]
            B, T, _ = xs.shape
            h = xs.new_zeros(B, self.hidden)
            logits, gates, recs = [], [], []
            for t in range(T):
                fused, g = self.fuse(xs[:, t])
                h = self.gru(fused, h)
                out = torch.cat([h, fused], -1)
                logits.append(self.head(out).squeeze(-1))
                gates.append(g)
                recs.append(self.recon(out))
            return torch.stack(logits, 1), torch.stack(gates, 1), torch.stack(recs, 1)

    return FusionGRU()


def export_fusion_onnx(model, path: str) -> None:
    """Xuất MỘT BƯỚC streaming: x[1,D], h[1,H] -> prob[1], h_out[1,H], gates[1,M] (đúng tên backend đọc)."""
    import torch

    class Step(torch.nn.Module):
        def __init__(self, m):
            super().__init__()
            self.m = m

        def forward(self, x, h):
            return self.m.step(x, h)

    model.eval()
    step = Step(model)
    x = torch.zeros(1, DIM)
    h = torch.zeros(1, model.hidden)
    torch.onnx.export(step, (x, h), path, input_names=["x", "h"], output_names=["prob", "h_out", "gates"],
                      opset_version=17, dynamo=False)
    import onnx

    m = onnx.load(path)
    m.ir_version = min(m.ir_version, 9)  # onnxruntime trên server có thể là bản cũ
    m.metadata_props.add(key="schema_version", value=str(SCHEMA["version"]))
    m.metadata_props.add(key="features", value=",".join(FEATURES))
    onnx.save(m, path)


def check_onnx_parity(model, path: str, xs: np.ndarray, atol: float = 1e-4) -> float:
    """Chạy chuỗi bằng ONNX từng bước, so với PyTorch chạy cả chuỗi."""
    import onnxruntime as ort
    import torch

    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    model.eval()
    with torch.no_grad():
        ref = torch.sigmoid(model(torch.from_numpy(xs[None]).float())[0])[0].numpy()
    h = np.zeros((1, model.hidden), np.float32)
    got = []
    for x in xs:
        p, h, _ = sess.run(None, {"x": x[None].astype(np.float32), "h": h})
        got.append(float(np.asarray(p).reshape(-1)[0]))
    err = float(np.max(np.abs(np.array(got) - ref)))
    assert err < atol, err
    return err


# ----------------------------------------------------------------------------- phiên thi tổng hợp
def ks_window(kind: str, rng, ks_bona=None, ks_copy=None) -> np.ndarray:
    pool = ks_bona if kind == "bona" else ks_copy
    if pool is not None and len(pool):
        return pool[rng.integers(len(pool))]
    # mô phỏng: [keys, dwell_rel, dwell_cv, flight_rel, flight_cv, burst, pause, backspace, compositions, cps]
    if kind == "bona":
        k = rng.poisson(8)
        return np.array([k, rng.normal(1, .12), rng.uniform(.3, .7), rng.normal(1, .25), rng.uniform(.6, 1.4),
                         rng.uniform(.1, .4), rng.uniform(0, .15), rng.uniform(.03, .15), rng.poisson(1), k / 2], np.float32)
    k = rng.poisson(14)
    return np.array([k, rng.normal(.95, .08), rng.uniform(.2, .4), rng.normal(.8, .1), rng.uniform(.3, .7),
                     rng.uniform(.4, .8), rng.uniform(0, .03), rng.uniform(0, .03), rng.poisson(1), k / 2], np.float32)


EPISODES = ["tab_lookup", "phone", "second_person", "whisper", "paste", "second_screen", "llm_transcribe"]


def synth_session(sid: int, T: int, cheat: bool, tier: str, rng, ks_bona=None, ks_copy=None) -> dict:
    """Một phiên thi tổng hợp: nền bình thường có nhiễu thực tế + 1–3 đoạn gian lận nếu cheat."""
    FI = {n: i for i, n in enumerate(FEATURES)}
    ks = lambda kind: ks_window(kind, rng, ks_bona, ks_copy)  # noqa: E731
    V = np.zeros((T, len(FEATURES)), np.float32)
    y = np.zeros(T, np.int64)
    f = lambda name: FI[name]  # noqa: E731
    # --- nền bình thường
    miss_p = {"low": .12, "mid": .04, "high": .02}[tier]
    present = rng.random(T) > miss_p
    V[:, f("face_count_mean")] = present
    V[:, f("face_count_max")] = present
    V[:, f("face_missing_ratio")] = ~present
    V[:, f("yaw_mean")] = rng.normal(0, 8, T) * present
    V[:, f("yaw_std")] = np.abs(rng.normal(3, 2, T)) * present
    V[:, f("pitch_mean")] = rng.normal(5, 6, T) * present
    V[:, f("pitch_std")] = np.abs(rng.normal(2, 1.5, T)) * present
    glance = rng.random(T) < .02
    V[glance, f("yaw_mean")] = rng.choice([-1, 1], glance.sum()) * rng.uniform(30, 50, glance.sum())
    V[:, f("head_away_ratio")] = (np.abs(V[:, f("yaw_mean")]) > 30).astype(np.float32)
    V[:, f("box_shift")] = np.abs(rng.normal(.01, .01, T))
    V[:, f("face_size")] = rng.uniform(.06, .14) * present
    V[:, f("face_conf")] = rng.uniform(.8, .97, T) * present
    V[:, f("id_score")] = rng.uniform(.55, .8)
    V[:, f("obj_age")] = rng.uniform(0, 10, T)
    cough = rng.random(T) < .01
    V[:, f("speech_ratio")] = cough * rng.uniform(.1, .3, T)
    V[:, f("speech_segments")] = cough
    V[:, f("audio_energy")] = rng.uniform(.05, .2, T) + cough * .3
    notif = rng.random(T) < .004  # thông báo hệ thống làm mất focus ngắn
    V[notif, f("blur_sec")] = rng.uniform(.3, 1.5, notif.sum())
    typing = rng.random(T) < .35
    for t in np.where(typing)[0]:
        V[t, MOD_SLICES["keystroke"]] = ks("bona")
    V[:, f("question_switches")] = rng.random(T) < .03
    V[:, f("answer_changes")] = rng.random(T) < .05
    V[:, f("time_on_question")] = rng.uniform(0, 120, T)
    V[:, f("progress")] = np.linspace(0, 1, T)
    V[:, f("time_vs_expected")] = np.where(rng.random(T) < .05, rng.normal(0, 1, T), 0)
    lag = {"low": 120, "mid": 40, "high": 15}[tier]
    V[:, f("lag_ms")] = np.abs(rng.normal(lag, lag / 2, T))
    off = rng.random(T) < .003
    V[:, f("offline")] = off
    V[:, f("monitor_load")] = {"low": .12, "mid": .05, "high": .02}[tier]
    # --- phương thức có/không theo phiên
    masks = {"face": 1, "identity": int(rng.random() < .8), "object": int(rng.random() < .7), "audio": int(rng.random() < .85),
             "browser": 1, "keystroke": 1, "answer": 1, "system": 1}
    # --- đoạn gian lận
    kinds = []
    if cheat:
        for _ in range(rng.integers(1, 4)):
            kind = EPISODES[rng.integers(len(EPISODES))]
            L = int(rng.integers(5, 45))
            s0 = int(rng.integers(10, max(11, T - L - 1)))
            sl = slice(s0, s0 + L)
            y[sl] = 1
            kinds.append(kind)
            if kind == "tab_lookup":
                V[sl, f("hidden_sec")] = 2.0
                V[sl, f("tab_switches")] = 0
                V[s0, f("tab_switches")] = 1
                V[sl, f("face_count_mean")] = 1
                after = slice(s0 + L, min(T, s0 + L + 5))
                V[after, f("chars_after_return")] = rng.uniform(20, 80)
                for t in range(after.start, after.stop):
                    V[t, MOD_SLICES["keystroke"]] = ks("copy")
                y[after] = 1
            elif kind == "phone":
                V[sl, f("pitch_mean")] = rng.uniform(25, 45)
                V[sl, f("head_away_ratio")] = 1
                V[sl, f("obj_phone")] = rng.uniform(.4, .9, L) * (rng.random(L) < .6)
            elif kind == "second_person":
                V[sl, f("face_count_max")] = (rng.random(L) < .5) + 1
                V[sl, f("obj_person_extra")] = rng.uniform(.3, .9, L) * (rng.random(L) < .6)
                V[sl, f("speech_ratio")] = rng.uniform(.2, .7, L)
                V[sl, f("speech_segments")] = rng.integers(1, 3, L)
            elif kind == "whisper":
                V[sl, f("speech_ratio")] = rng.uniform(.1, .5, L)
                V[sl, f("audio_energy")] = rng.uniform(.15, .35, L)
            elif kind == "paste":
                V[s0, f("paste_attempts")] = rng.integers(1, 3)
                y[sl] = 0
                y[s0:s0 + 3] = 1
            elif kind == "second_screen":
                V[sl, f("screen_extended")] = 1
                V[sl, f("yaw_mean")] = rng.choice([-1, 1]) * rng.uniform(30, 55, L)
                V[sl, f("head_away_ratio")] = 1
                V[sl, f("obj_screen")] = rng.uniform(.3, .8, L) * (rng.random(L) < .4)
            elif kind == "llm_transcribe":
                for t in range(sl.start, sl.stop):
                    V[t, MOD_SLICES["keystroke"]] = ks("copy")
                V[sl, f("time_vs_expected")] = rng.uniform(-4, -2)
    X = np.stack([model_input(dict(zip(FEATURES, row)), masks) for row in V])
    return {"id": f"syn:{sid}", "X": X, "y": y, "tier": tier, "source": "synthetic", "kinds": kinds}




def make_synthetic_sessions(n: int, T: int, rng, cheat_rate: float = 0.5, tiers=("low", "mid", "high"),
                            tier_p=(0.3, 0.5, 0.2), ks_bona=None, ks_copy=None) -> list[dict]:
    return [synth_session(i, T, cheat=rng.random() < cheat_rate, tier=rng.choice(list(tiers), p=list(tier_p)), rng=rng,
                          ks_bona=ks_bona, ks_copy=ks_copy) for i in range(n)]

## 1. Lưu kết quả lên Google Drive (khuyến nghị)
Notebook 03 và 04 đọc file `keystroke_windows.parquet` tạo ở đây. Bỏ qua ô này nếu không dùng Drive.

In [ ]:
import os
from pathlib import Path

OUT = Path("/content/examguard")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/examguard")
except Exception as e:  # chạy ngoài Colab
    print("Không mount được Drive:", e)
OUT.mkdir(parents=True, exist_ok=True)
DATA = Path("/content/data") if Path("/content").exists() else Path("data")
DATA.mkdir(parents=True, exist_ok=True)
print("Kết quả lưu tại", OUT)

## 2. Tải dữ liệu
Bộ tiếng Việt: thư mục `Attack4` (gồm file bình thường và file đã thao túng). Bộ tiếng Anh: clone repo kèm Git LFS.
Nếu Google Drive báo quá lượt tải, mở link trong trình duyệt, "Add shortcut to Drive" rồi đặt `VI_DIR` trỏ tới đó.

In [ ]:
import subprocess, zipfile

VI_DIR = DATA / "Attack4"
if not any(VI_DIR.glob("User*")):
    subprocess.run(["gdown", "--folder", "https://drive.google.com/drive/folders/1NdT6VrEq26uF2DigYAlRVg1wT-xFlSHT",
                    "-O", str(VI_DIR)], check=False)
print("Tiếng Việt:", len(list(VI_DIR.glob("User*"))), "người")

EN_REPO = DATA / "keystroke-llm-plagiarism"
if not EN_REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ijcb-2024/keystroke-llm-plagiarism", str(EN_REPO)], check=False)
    subprocess.run(["git", "lfs", "pull"], cwd=EN_REPO, check=False)
EN_DIR = DATA / "en"
for z in (EN_REPO / "Datasets").glob("*.zip"):
    if z.stat().st_size < 1000:
        print("⚠️", z.name, "vẫn là con trỏ Git LFS (hết băng thông LFS?). Tải tay từ GitHub và giải nén vào", EN_DIR)
        continue
    with zipfile.ZipFile(z) as f:
        f.extractall(EN_DIR / z.stem.replace(" ", "_"))
print("Tiếng Anh:", sorted(str(p.relative_to(EN_DIR)) for p in EN_DIR.rglob("*") if p.is_file())[:20])

## 3. Đọc dữ liệu về một định dạng chung
Mỗi **mẫu** = một câu trả lời: `(ngôn ngữ, người, câu hỏi, nhãn, danh sách sự kiện phím)`; sự kiện = `(t_ms, 'down'|'up', code, ime)`.

- Tiếng Việt: `{"keystrokes": [{key, code, event, timestamp, question_index "phiên.câu"}]}`; phiên 1 = B, 2 = P, 3 = T;
  file `*_attack.json`: phiên 2 = FP, 3 = FT. Phím ảo của Unikey đã được tác giả loại bỏ.
- Tiếng Anh: `{id: {"keyboard_data": [[KD|KU, phím, t], ...]}}` (một số file lưu giá trị là chuỗi JSON, một số đảo thứ tự cột);
  `free` = tự viết (B), `fixed` = chép văn bản cho sẵn (T).

In [ ]:
import json
from collections import defaultdict

import numpy as np
import pandas as pd

from eg_common import session_keystroke_windows, summarize

VI_LABEL = {("normal", "1"): "B", ("normal", "2"): "P", ("normal", "3"): "T", ("attack", "2"): "FP", ("attack", "3"): "FT"}


def load_vietnamese(root: Path) -> list[dict]:
    samples = []
    for user_dir in sorted(root.glob("User*")):
        for f in sorted(user_dir.glob("*.json")):
            kind = "attack" if "attack" in f.name else "normal"
            phase = "2" if f.name.startswith("second") else "1"
            data = json.loads(f.read_text(encoding="utf-8"))
            by_q = defaultdict(list)
            for e in data.get("keystrokes", []):
                qi = str(e.get("question_index", ""))
                if "." not in qi:
                    continue
                sess, q = qi.split(".", 1)
                lab = VI_LABEL.get((kind, sess))
                if lab is None or (kind == "attack" and sess == "1"):
                    continue
                typ = "down" if str(e.get("event", "")).lower() == "keydown" else "up"
                by_q[(lab, q)].append((float(e["timestamp"]), typ, e.get("code") or e.get("key"), False))
            for (lab, q), ev in by_q.items():
                samples.append({"lang": "vi", "user": user_dir.name, "question": f"{phase}.{q}", "label": lab, "events": ev})
    return samples


def _rows(v):
    if isinstance(v, str):
        v = json.loads(v)
    if isinstance(v, dict):
        v = v.get("keyboard_data", v.get("data", []))
    return v


def load_english(root: Path) -> list[dict]:
    samples = []
    for f in sorted(root.rglob("*")):
        if not f.is_file() or f.suffix.lower() not in (".json", ".txt"):
            continue
        name = f.name.lower()
        if "fixed" in name:
            lab = "T"
        elif "free" in name:
            lab = "B"
        else:
            continue
        try:
            data = json.loads(f.read_text(encoding="utf-8", errors="ignore"))
        except json.JSONDecodeError:
            continue
        source = "buffalo" if "buffalo" in name else "sbu" if "raw_temp" in name else "proposed"
        for key, val in data.items():
            ev = []
            for r in _rows(val):
                if len(r) < 3:
                    continue
                a, b = str(r[0]).upper(), str(r[1]).upper()
                if a in ("KD", "KU"):
                    typ, k = a, r[1]
                elif b in ("KD", "KU"):
                    typ, k = b, r[0]
                else:
                    continue
                ev.append((float(r[2]), "down" if typ == "KD" else "up", str(k).lower(), False))
            if len(ev) >= 20:
                user = f"{source}:" + str(key).split("_")[0]
                samples.append({"lang": "en", "user": user, "question": str(key), "label": lab, "events": ev, "source": source})
    return samples


vi = load_vietnamese(VI_DIR)
en = load_english(EN_DIR)
samples = vi + en
df_s = pd.DataFrame([{k: v for k, v in s.items() if k != "events"} | {"n_events": len(s["events"])} for s in samples])
print(df_s.groupby(["lang", "label"]).agg(samples=("user", "size"), users=("user", "nunique"), events=("n_events", "median")))

## 4. Cắt thành cửa sổ 2 giây bằng đúng công thức của trình duyệt
`epsilon=inf`: chưa thêm nhiễu (notebook 04 đánh giá ảnh hưởng của nhiễu Laplace lên độ chính xác và quyền riêng tư).

In [ ]:
from eg_common import FEATURES, MOD_OF

KFEATS = [f for f in FEATURES if MOD_OF[f] == "keystroke"]
rows = []
for sid, s in enumerate(samples):
    ws = session_keystroke_windows(s["events"], warmup_s=20, epsilon=float("inf"))
    for t, w in enumerate(ws):
        if w["keys"] == 0:
            continue
        rows.append({"sample": sid, "t": t, "lang": s["lang"], "user": s["user"], "label": s["label"], **w})
win = pd.DataFrame(rows)
win.to_parquet(OUT / "keystroke_windows.parquet")
print(win.shape, "->", OUT / "keystroke_windows.parquet")
win.groupby("label")[KFEATS].mean().round(3)

## 5. Baseline: XGBoost trên đặc trưng tổng hợp mỗi câu trả lời
Nhãn nhị phân: **gian lận = có hỗ trợ LLM (P, T, FP, FT)**, bình thường = B. Chia theo người (GroupKFold).

In [ ]:
from sklearn.model_selection import GroupKFold
from xgboost import XGBClassifier

POS = {"P", "T", "FP", "FT"}


def aggregate(w: pd.DataFrame) -> pd.DataFrame:
    g = w.groupby("sample")
    agg = g[KFEATS].agg(["mean", "std", "median"]).fillna(0)
    agg.columns = [f"{a}_{b}" for a, b in agg.columns]
    meta = g[["lang", "user", "label"]].first()
    return agg.join(meta).join(g.size().rename("n_windows"))


A = aggregate(win)
FEAT_COLS = [c for c in A.columns if c not in ("lang", "user", "label")]


def fit_xgb(tr: pd.DataFrame, seed=0) -> XGBClassifier:
    clf = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                        random_state=seed, eval_metric="logloss")
    return clf.fit(tr[FEAT_COLS], tr.label.isin(POS).astype(int))


def cv_xgb(train_df: pd.DataFrame, test_df: pd.DataFrame | None = None, train_labels=("B", "P", "T"), folds=5, seed=0):
    """test_df=None: kiểm tra chéo theo NGƯỜI trên train_df (fold kiểm tra giữ mọi nhãn của người đó).
    test_df có giá trị: huấn luyện trên toàn bộ train_df, kiểm tra trên test_df (chéo ngôn ngữ).
    Trả về (y, điểm, nhãn gốc)."""
    tr_all = train_df[train_df.label.isin(train_labels)]
    if test_df is not None:
        clf = fit_xgb(tr_all, seed)
        return test_df.label.isin(POS).astype(int).values, clf.predict_proba(test_df[FEAT_COLS])[:, 1], test_df.label.values
    ys, ss, labs = [], [], []
    users = np.array(sorted(train_df.user.unique()))
    for tr_u, te_u in GroupKFold(n_splits=min(folds, len(users))).split(users, groups=users):
        clf = fit_xgb(tr_all[tr_all.user.isin(users[tr_u])], seed)
        te = train_df[train_df.user.isin(users[te_u])]
        ys.append(te.label.isin(POS).astype(int).values)
        ss.append(clf.predict_proba(te[FEAT_COLS])[:, 1])
        labs.append(te.label.values)
    return np.concatenate(ys), np.concatenate(ss), np.concatenate(labs)


results = []
A_main = A[A.label.isin(["B", "P", "T"])]  # FP/FT để riêng cho thí nghiệm né tránh (mục 6)
for lang in ("vi", "en"):
    sub = A_main[A_main.lang == lang]
    if len(sub) < 20:
        continue
    y, s, lab = cv_xgb(sub)
    results.append({"setting": f"riêng {lang}", "model": "XGBoost", **summarize(y, s)})
y, s, lab = cv_xgb(A_main)
results.append({"setting": "chung vi+en", "model": "XGBoost", **summarize(y, s)})
if (A_main.lang == "vi").any() and (A_main.lang == "en").any():
    for a, b in (("vi", "en"), ("en", "vi")):
        y, s, _ = cv_xgb(A_main[A_main.lang == a], A_main[A_main.lang == b])
        results.append({"setting": f"chéo {a}→{b}", "model": "XGBoost", **summarize(y, s)})
pd.DataFrame(results)

## 6. Chống né tránh: huấn luyện B/P/T, kiểm tra trên FP/FT (tiếng Việt)
FP/FT là câu trả lời có hỗ trợ LLM nhưng **nhịp gõ bị thao túng** (đổi tốc độ, chèn sửa xóa, chèn ngập ngừng).
Recall giảm mạnh ở đây là lý do cần hợp nhất nhiều phương thức (C1) thay vì chỉ dựa vào keystroke.

In [ ]:
vi_A = A[A.lang == "vi"]
evasion = []
for train_labels in (("B", "P", "T"), ("B", "P", "T", "FP", "FT")):
    y, s, lab = cv_xgb(vi_A, train_labels=train_labels)
    # kiểm tra trên toàn bộ nhãn của người ở fold kiểm tra
    for target in ("P", "T", "FP", "FT"):
        m = np.isin(lab, ["B", target])
        evasion.append({"train": "+".join(train_labels), "test": f"B vs {target}", **summarize(y[m], s[m])})
pd.DataFrame(evasion)

## 7. Mô hình chuỗi (TCN nhỏ) trên chuỗi cửa sổ 2 giây
Đầu vào là chuỗi vector keystroke (10 chiều) — đúng thứ mô hình hợp nhất nhận ở notebook 03.

In [ ]:
import torch
from torch import nn

DEV = "cuda" if torch.cuda.is_available() else "cpu"
SCALE = np.array([10, 1, 1, 1, 1, 1, 1, 0.3, 3, 5], np.float32)  # khớp "scale" trong feature_schema.json
MAXLEN = 120


def to_seq(w: pd.DataFrame):
    X, y, users, labs, langs = [], [], [], [], []
    for sid, g in w.groupby("sample"):
        x = np.clip(g.sort_values("t")[KFEATS].values.astype(np.float32) / SCALE, -5, 5)[:MAXLEN]
        pad = np.zeros((MAXLEN, len(KFEATS)), np.float32)
        pad[:len(x)] = x
        X.append(pad)
        y.append(int(g.label.iloc[0] in POS))
        users.append(g.user.iloc[0])
        labs.append(g.label.iloc[0])
        langs.append(g.lang.iloc[0])
    return np.stack(X), np.array(y), np.array(users), np.array(labs), np.array(langs)


class TCN(nn.Module):
    def __init__(self, d_in=len(KFEATS), d=48):
        super().__init__()
        layers, c = [], d_in
        for dil in (1, 2, 4, 8):
            layers += [nn.Conv1d(c, d, 3, padding=dil, dilation=dil), nn.GELU(), nn.Dropout(0.1)]
            c = d
        self.net = nn.Sequential(*layers)
        self.out = nn.Linear(d, 1)

    def forward(self, x):  # [B, T, F]
        mask = (x.abs().sum(-1, keepdim=True) > 0).float()
        h = self.net(x.transpose(1, 2)).transpose(1, 2) * mask
        return self.out(h.sum(1) / mask.sum(1).clamp(min=1)).squeeze(-1)


def train_tcn(X, y, epochs=40, seed=0):
    torch.manual_seed(seed)
    m = TCN().to(DEV)
    opt = torch.optim.AdamW(m.parameters(), 2e-3, weight_decay=1e-3)
    Xt, yt = torch.tensor(X).to(DEV), torch.tensor(y, dtype=torch.float32).to(DEV)
    pos_w = torch.tensor((1 - y.mean()) / max(y.mean(), 1e-3), device=DEV)
    for _ in range(epochs):
        m.train()
        for idx in torch.randperm(len(Xt)).split(64):
            loss = nn.functional.binary_cross_entropy_with_logits(m(Xt[idx]), yt[idx], pos_weight=pos_w)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return m.eval()


Xs, ys, us, ls, langs = to_seq(win)
tcn_rows = []
for lang in ("vi", "en", "all"):
    sel = (langs == lang) if lang != "all" else np.ones(len(ys), bool)
    sel &= np.isin(ls, ["B", "P", "T"])
    if sel.sum() < 30:
        continue
    yy, ss = [], []
    for tr, te in GroupKFold(min(5, len(set(us[sel])))).split(Xs[sel], groups=us[sel]):
        m = train_tcn(Xs[sel][tr], ys[sel][tr])
        with torch.no_grad():
            ss.append(torch.sigmoid(m(torch.tensor(Xs[sel][te]).to(DEV))).cpu().numpy())
        yy.append(ys[sel][te])
    tcn_rows.append({"setting": lang, "model": "TCN", **summarize(np.concatenate(yy), np.concatenate(ss))})
pd.DataFrame(tcn_rows)

## 8. Lưu kết quả
- `keystroke_windows.parquet`: cửa sổ 2 giây có nhãn (dùng ở notebook 03 để chèn đoạn "chép bài" vào phiên thi tổng hợp,
  và notebook 04 cho thí nghiệm tái nhận dạng người dùng).
- `results_keystroke.csv`: bảng số liệu cho bài báo.

In [ ]:
pd.concat([pd.DataFrame(results), pd.DataFrame(evasion), pd.DataFrame(tcn_rows)]).to_csv(OUT / "results_keystroke.csv", index=False)
print("Đã lưu", OUT / "results_keystroke.csv")